In [2]:
import os
import shutil
import random

real_path = r"D:\python\python\testing\dataset\Human_Faces_Dataset\Real_Images"
ai_path = r"D:\python\python\testing\dataset\Human_Faces_Dataset\AI_Generated_Images"

base_output = r"D:\python\python\testing\dataset\Human_Faces_Dataset\split_data"

train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

classes = {
    "Real_Images": real_path,
    "AI_Generated_Images": ai_path
}

# Delete old folder first
if os.path.exists(base_output):
    shutil.rmtree(base_output)

def create_folders():
    for split in ["train", "val", "test"]:
        for class_name in classes.keys():
            folder_path = os.path.join(base_output, split, class_name)
            os.makedirs(folder_path, exist_ok=True)

def split_and_copy(source_folder, class_name):
    files = os.listdir(source_folder)

    # support more image types
    files = [
        f for f in files
        if f.lower().endswith(
            (".png", ".jpg", ".jpeg", ".webp", ".jfif", ".bmp")
        )
    ]

    print(f"{class_name} Total Images Found: {len(files)}")

    random.shuffle(files)

    total = len(files)

    train_end = int(total * train_ratio)
    val_end = train_end + int(total * val_ratio)

    train_files = files[:train_end]
    val_files = files[train_end:val_end]
    test_files = files[val_end:]

    split_map = {
        "train": train_files,
        "val": val_files,
        "test": test_files
    }

    for split_name, split_files in split_map.items():
        for file_name in split_files:
            src = os.path.join(source_folder, file_name)
            dst = os.path.join(base_output, split_name, class_name, file_name)

            try:
                shutil.copy2(src, dst)
            except Exception as e:
                print(f"Error copying {file_name}: {e}")

    print(f"\n{class_name} Split Complete")
    print(f"Train: {len(train_files)}")
    print(f"Validation: {len(val_files)}")
    print(f"Test: {len(test_files)}")
    print("-" * 40)

create_folders()

for class_name, folder_path in classes.items():
    split_and_copy(folder_path, class_name)

print("Dataset split completed successfully.")

Real_Images Total Images Found: 5000

Real_Images Split Complete
Train: 3500
Validation: 750
Test: 750
----------------------------------------
AI_Generated_Images Total Images Found: 4630

AI_Generated_Images Split Complete
Train: 3241
Validation: 694
Test: 695
----------------------------------------
Dataset split completed successfully.


In [1]:
import os

real_path = r"D:\python\python\testing\dataset\Human_Faces_Dataset\Real_Images"
ai_path = r"D:\python\python\testing\dataset\Human_Faces_Dataset\AI_Generated_Images"

print("Real path exists:", os.path.exists(real_path))
print("AI path exists:", os.path.exists(ai_path))

print("Real files count:", len(os.listdir(real_path)))
print("AI files count:", len(os.listdir(ai_path)))

Real path exists: True
AI path exists: True
Real files count: 5000
AI files count: 4630


In [12]:
import tensorflow as tf

train_dir = r"D:\python\python\testing\dataset\Human_Faces_Dataset\split_data\train"
val_dir = r"D:\python\python\testing\dataset\Human_Faces_Dataset\split_data\val"
test_dir = r"D:\python\python\testing\dataset\Human_Faces_Dataset\split_data\test"



train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(224,224),
    batch_size=32,
    color_mode="rgb"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=(224,224),
    batch_size=32,
    color_mode="rgb"
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(224,224),
    batch_size=32,
    shuffle=False,
    color_mode="rgb"
)

print('complted')

Found 6741 files belonging to 2 classes.
Found 1444 files belonging to 2 classes.
Found 1445 files belonging to 2 classes.
complted


Model Train and Test

In [13]:
import tensorflow as tf
from  tensorflow.keras import layers,models
from tensorflow.keras.applications import MobileNetV2

In [14]:
# for image more understanders

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])


# for base model

base_model = MobileNetV2(input_shape=(224,224,3),weights='imagenet',include_top=False)
base_model.trainable = False

In [ ]:
model = models.Sequential([
    data_augmentation,

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(128,activation='relu'),

    layers.Dropout(0.5),

    layers.Dense(2,activation='softmax')
])

model.compile(
    optimizer = 'adam',

    loss = 'sparse_categorical_crossentropy',

    metrics = ['accuracy']
)

In [18]:
model.fit(train_ds,validation_data=val_ds,epochs=10)

Epoch 1/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 191s 876ms/step - accuracy: 0.9356 - loss: 0.1580 - val_accuracy: 0.9799 - val_loss: 0.0587
Epoch 2/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 162s 770ms/step - accuracy: 0.9672 - loss: 0.0837 - val_accuracy: 0.9688 - val_loss: 0.0837
Epoch 3/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 161s 765ms/step - accuracy: 0.9761 - loss: 0.0678 - val_accuracy: 0.9855 - val_loss: 0.0392
Epoch 4/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 163s 773ms/step - accuracy: 0.9764 - loss: 0.0663 - val_accuracy: 0.9868 - val_loss: 0.0325
Epoch 5/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 164s 775ms/step - accuracy: 0.9800 - loss: 0.0599 - val_accuracy: 0.9889 - val_loss: 0.0322
Epoch 6/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 162s 766ms/step - accuracy: 0.9825 - loss: 0.0516 - val_accuracy: 0.9855 - val_loss: 0.0303
Epoch 7/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 200s 753ms/step - accuracy: 0.9772 - loss: 0.0611 - val_accuracy: 0.9820 - val_loss: 0.0496
Epoch 8/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 158s 747ms/step - accuracy: 0.9809 -

In [19]:
model.evaluate(test_ds)

46/46 ━━━━━━━━━━━━━━━━━━━━ 26s 569ms/step - accuracy: 0.9896 - loss: 0.0247


[0.024740993976593018, 0.9896193742752075]

In [20]:
model.save("face_classifier.h5")

In [21]:
from tensorflow.keras.models import load_model

model = load_model("face_classifier.h5")

In [23]:
import lz4
print("lz4 installed successfully")

lz4 installed successfully


In [24]:
# predict.py

import cv2
import numpy as np
from mtcnn import MTCNN
from tensorflow.keras.models import load_model

# -----------------------------
# 1. Load your trained model
# -----------------------------
model = load_model("face_classifier.h5")

# -----------------------------
# 2. Load MTCNN face detector
# -----------------------------
detector = MTCNN()

# -----------------------------
# 3. Class labels
# IMPORTANT:
# Check your actual class order
# from training dataset
# -----------------------------
class_names = [
    "AI Generated Face",
    "Human Face"
]

# -----------------------------
# 4. Read input image
# -----------------------------
img = cv2.imread("myphoto.jpg")

if img is None:
    print("Image not found")
    exit()

# Keep original for drawing
output = img.copy()

# -----------------------------
# 5. Convert BGR to RGB
# -----------------------------
rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# -----------------------------
# 6. Detect faces
# -----------------------------
faces = detector.detect_faces(rgb_img)

if len(faces) == 0:
    print("No face detected")

for face_data in faces:
    x, y, w, h = face_data["box"]

    # Fix negative values if any
    x = max(0, x)
    y = max(0, y)

    # -----------------------------
    # 7. Crop face
    # -----------------------------
    face_crop = img[y:y+h, x:x+w]

    if face_crop.size == 0:
        continue

    # -----------------------------
    # 8. Resize to model input size
    # -----------------------------
    face_crop = cv2.resize(face_crop, (224, 224))

    # -----------------------------
    # 9. Normalize
    # -----------------------------
    face_crop = face_crop.astype("float32") / 255.0

    # -----------------------------
    # 10. Add batch dimension
    # Shape becomes:
    # (1, 224, 224, 3)
    # -----------------------------
    face_crop = np.expand_dims(face_crop, axis=0)

    # -----------------------------
    # 11. Predict
    # -----------------------------
    prediction = model.predict(face_crop, verbose=0)

    predicted_class = np.argmax(prediction)
    confidence = np.max(prediction) * 100

    label = class_names[predicted_class]

    display_text = f"{label} ({confidence:.2f}%)"

    # -----------------------------
    # 12. Draw bounding box + label
    # -----------------------------
    cv2.rectangle(output, (x, y), (x+w, y+h), (0, 255, 0), 2)

    cv2.putText(
        output,
        display_text,
        (x, y - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2
    )

# -----------------------------
# 13. Show final result
# -----------------------------
cv2.imshow("Human vs AI Face Detection", output)

cv2.waitKey(0)
cv2.destroyAllWindows()

ValueError: LZ4 is not installed. Install it with pip: https://python-lz4.readthedocs.io/

In [27]:
detector = MTCNN()

ValueError: LZ4 is not installed. Install it with pip: https://python-lz4.readthedocs.io/

In [1]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# Load your trained model
model = load_model(r"D:\python\python\testing\face_classifier.h5")

# OpenCV face detector
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

# Start webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()

    if not ret:
        print("Camera not working")
        break

    # Convert to gray for face detection
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect faces
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5
    )

    for (x, y, w, h) in faces:
        # Crop face
        face = frame[y:y+h, x:x+w]

        # Resize for model
        face_resized = cv2.resize(face, (224,224))

        # Normalize
        face_array = face_resized / 255.0
        face_array = np.expand_dims(face_array, axis=0)

        # Prediction
        prediction = model.predict(face_array, verbose=0)

        if prediction[0][0] > 0.5:
            label = "REAL FACE"
        else:
            label = "AI GENERATED"

        # Draw rectangle
        cv2.rectangle(
            frame,
            (x, y),
            (x + w, y + h),
            (0, 255, 0),
            2
        )

        # Put label
        cv2.putText(
            frame,
            label,
            (x, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

    cv2.imshow("Face Detection", frame)

    # Press q to quit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

In [18]:
import cv2
import numpy as np

video = cv2.VideoCapture(0)

face_detecte = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')

face_datas = []

i1=0
try:
    while True:
        ret, frame = video.read()
        gray = cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
        myface = face_detecte.detectMultiScale(gray,scaleFactor=1.3,minNeighbors=5)
        for (x,y,w,h) in myface:
            print(f'i1:{i1},x:{x},y:{y},w:{w},h:{h}')
            copy_image = frame[y:y+h,x:x+w,:]
            resize_img = cv2.resize(copy_image,dsize=(100,100))
            if len(face_datas)<=10 and i1%10==0:
                face_datas.append(resize_img)
            i1+=1
            cv2.putText(frame, str(len(face_datas)), (10, 30), cv2.FONT_HERSHEY_COMPLEX, 1, (0,0,255), 1)
            cv2.rectangle(frame,pt1=(x,y),pt2=(x+w,y+h),color=(255,0,0),thickness=1)

        cv2.imshow('frame',frame)
        if not ret or cv2.waitKey(1)&0xFF == ord('s'):
            break
        

    video.release()
except Exception:
    print(Exception)
    cv2.destroyAllWindows()
cv2.destroyAllWindows()

i1:0,x:192,y:236,w:189,h:189
i1:1,x:192,y:236,w:189,h:189
i1:2,x:192,y:236,w:189,h:189
i1:3,x:192,y:236,w:189,h:189
i1:4,x:181,y:230,w:188,h:188
i1:5,x:181,y:230,w:188,h:188
i1:6,x:181,y:230,w:188,h:188
i1:7,x:181,y:230,w:188,h:188
i1:8,x:183,y:225,w:176,h:176
i1:9,x:183,y:225,w:176,h:176
i1:10,x:183,y:225,w:176,h:176
i1:11,x:183,y:225,w:176,h:176
i1:12,x:196,y:229,w:153,h:153
i1:13,x:196,y:229,w:153,h:153
i1:14,x:196,y:229,w:153,h:153
i1:15,x:196,y:229,w:153,h:153
i1:16,x:230,y:218,w:158,h:158
i1:17,x:230,y:218,w:158,h:158
i1:18,x:230,y:218,w:158,h:158
i1:19,x:208,y:207,w:175,h:175
i1:20,x:208,y:207,w:175,h:175
i1:21,x:208,y:207,w:175,h:175
i1:22,x:208,y:207,w:175,h:175
i1:23,x:184,y:199,w:190,h:190
i1:24,x:184,y:199,w:190,h:190
i1:25,x:179,y:202,w:183,h:183
i1:26,x:179,y:202,w:183,h:183
i1:27,x:179,y:202,w:183,h:183
i1:28,x:171,y:198,w:192,h:192
i1:29,x:171,y:198,w:192,h:192
i1:30,x:171,y:198,w:192,h:192
i1:31,x:171,y:198,w:192,h:192
i1:32,x:173,y:202,w:185,h:185
i1:33,x:173,y:202,w:

In [19]:
lenth = len(face_datas)
while True:

    print(lenth)
    for i in range(len(face_datas)):
        print(i)
        if i<lenth-2:
            print(face_datas[i])
            cv2.namedWindow('mywindown',cv2.WINDOW_NORMAL)
            cv2.resizeWindow('mywindown',500,500)
            cv2.imshow('mywindown',face_datas[i])
            cv2.waitKey(1000)
        break
        
    if cv2.waitKey(1)&0xFF == ord('s'):
        break

cv2.destroyAllWindows()

11
0
[[[ 48  48  48]
  [ 47  47  47]
  [ 46  46  46]
  ...
  [ 54  54  54]
  [ 54  54  54]
  [ 54  54  54]]

 [[ 47  47  47]
  [ 45  45  45]
  [ 44  44  44]
  ...
  [ 55  55  55]
  [ 54  54  54]
  [ 54  54  54]]

 [[ 46  46  46]
  [ 45  45  45]
  [ 44  44  44]
  ...
  [ 56  56  56]
  [ 53  53  53]
  [ 53  53  53]]

 ...

 [[ 96 100  99]
  [ 97  99  99]
  [ 98  98  98]
  ...
  [ 76  76  76]
  [ 75  75  75]
  [ 75  75  75]]

 [[ 93  99  98]
  [ 94  99  98]
  [ 95  97  97]
  ...
  [ 75  75  75]
  [ 75  75  75]
  [ 76  76  76]]

 [[ 92 100  99]
  [ 92  99  98]
  [ 94 100  98]
  ...
  [ 74  74  74]
  [ 76  76  76]
  [ 76  76  77]]]
11
0
[[[ 48  48  48]
  [ 47  47  47]
  [ 46  46  46]
  ...
  [ 54  54  54]
  [ 54  54  54]
  [ 54  54  54]]

 [[ 47  47  47]
  [ 45  45  45]
  [ 44  44  44]
  ...
  [ 55  55  55]
  [ 54  54  54]
  [ 54  54  54]]

 [[ 46  46  46]
  [ 45  45  45]
  [ 44  44  44]
  ...
  [ 56  56  56]
  [ 53  53  53]
  [ 53  53  53]]

 ...

 [[ 96 100  99]
  [ 97  99  99]
  [ 98  98

KeyboardInterrupt: 

In [15]:
lenth = len(face_datas)
print(lenth)
for i in range(len(face_datas)):
    if i==lenth-1:
        print(face_datas[i])

11
[[[ 98  97  99]
  [ 96  96  98]
  [ 97  97  97]
  ...
  [ 49  51  51]
  [ 50  52  52]
  [ 50  52  52]]

 [[102  98  99]
  [100  94  95]
  [ 94  92  92]
  ...
  [ 52  52  53]
  [ 52  52  53]
  [ 52  52  52]]

 [[100  98  98]
  [ 99  95  96]
  [ 92  91  91]
  ...
  [ 52  51  53]
  [ 51  50  52]
  [ 51  52  51]]

 ...

 [[209 209 209]
  [205 205 205]
  [203 203 203]
  ...
  [118 118 118]
  [118 118 118]
  [118 118 118]]

 [[212 212 212]
  [206 206 206]
  [199 199 199]
  ...
  [118 118 118]
  [120 120 120]
  [122 122 122]]

 [[207 207 207]
  [201 201 201]
  [199 199 199]
  ...
  [129 129 129]
  [131 131 131]
  [135 135 135]]]


In [ ]:
import cv2
import pytesseract

img = cv2.imread('orc.webp')

gray = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)

text = pytesseract.image_to_string(gray)

print(text)

TesseractNotFoundError: tesseract is not installed or it's not in your PATH. See README file for more information.

In [5]:
import pytesseract
from PIL import Image

# Load the image
img = Image.open('orc.webp')

# Extract text
text = pytesseract.image_to_string(img)
print(text)

TesseractNotFoundError: tesseract is not installed or it's not in your PATH. See README file for more information.

In [ ]:
import easyocr


reader = easyocr.Reader(['en'])
text = reader.readtext('orc.webp',detail=0)
print(text)

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

C:\Users\EATSK\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


['You', 'can use this tooi', 'to write a long text and', 'save it as', 'an', 'image:', 'L', '9']


In [8]:
print(reader.readtext('orc.webp'))

C:\Users\EATSK\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[([[np.int32(56), np.int32(177)], [np.int32(260), np.int32(177)], [np.int32(260), np.int32(278)], [np.int32(56), np.int32(278)]], 'You', np.float64(0.999914381549895)), ([[np.int32(280), np.int32(174)], [np.int32(1127), np.int32(174)], [np.int32(1127), np.int32(279)], [np.int32(280), np.int32(279)]], 'can use this tooi', np.float64(0.4018649559556144)), ([[np.int32(47), np.int32(289)], [np.int32(1237), np.int32(289)], [np.int32(1237), np.int32(426)], [np.int32(47), np.int32(426)]], 'to write a long text and', np.float64(0.8206421207281374)), ([[np.int32(52), np.int32(426)], [np.int32(546), np.int32(426)], [np.int32(546), np.int32(526)], [np.int32(52), np.int32(526)]], 'save it as', np.float64(0.9275948859814601)), ([[np.int32(569), np.int32(433)], [np.int32(701), np.int32(433)], [np.int32(701), np.int32(523)], [np.int32(569), np.int32(523)]], 'an', np.float64(0.9999942680334141)), ([[np.int32(714), np.int32(417)], [np.int32(1081), np.int32(417)], [np.int32(1081), np.int32(553)], [np.in